Maybe try to read in the opera datasets into MintPy for improved time series velocities and ascending/descending decomposition? Then clip the datasets by the aquifers for zonal statistics?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from mintpy import view, tsview
import os
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd
from transboundary_opera.decomposer import InSARDecomposer
from transboundary_opera.decomposition_tools import InSARDecomposer as test_decomposer

import h5py
from mintpy.asc_desc2horz_vert import get_overlap_lalo
from mintpy.utils import readfile
import numpy as np

In [ ]:
gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")


data_storage = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"    # linux
# data_storage = Path('/mnt/c/Users/clayc/Documents/US_Mex_InSAR') / "OPERA_data"    # WSL
mintpy_paths = list(data_storage.glob('*/mintpy'))
opera_product_paths = list(data_storage.glob('*/subset-ncs/*.nc'))

In [ ]:

def get_asc_desc_pairs(data_dir):
    asc_list = []
    desc_list = []
    
    for ts_file in list(data_dir.glob('*/*/timeseries.h5')):
        orb_dir = dict(h5py.File(ts_file, 'r').attrs)['ORBIT_DIRECTION']
    
        if orb_dir == 'Ascending':
            asc_list.append(ts_file)
        else:
            desc_list.append(ts_file)

    # Create a list to store overlapping pairs
    overlapping_pairs = []

    # Get attributes for all files
    asc_attrs = [readfile.read_attribute(str(f)) for f in asc_list]
    desc_attrs = [readfile.read_attribute(str(f)) for f in desc_list]

    # Check each ascending track against each descending track
    for i, asc_file in enumerate(asc_list):
        for j, desc_file in enumerate(desc_list):
            try:
                # Check if there's an overlap
                bbox = get_overlap_lalo([asc_attrs[i], desc_attrs[j]])
                if bbox is not None:
                    overlapping_pairs.append({
                        'asc_file': asc_file,
                        'desc_file': desc_file,
                        'asc_frame': asc_file.parent.parent.name,
                        'desc_frame': desc_file.parent.parent.name,
                        'bbox': bbox
                    })
            except:
                continue
            
    return overlapping_pairs

def plot_displacements(horz_file, vert_file, time_idx=None, vlim=None, cmap='RdBu'):
    """
    Plot horizontal and vertical displacement maps.
    
    Parameters
    ----------
    horz_file : str or Path
        Path to horizontal displacement HDF5 file
    vert_file : str or Path
        Path to vertical displacement HDF5 file
    time_idx : int, optional
        Time index to plot. If None, plots all time steps.
    vlim : tuple, optional
        Color limits as (vmin, vmax). If None, uses symmetric limits.
    cmap : str
        Colormap name (default: 'RdBu')
    
    Returns
    -------
    fig : matplotlib Figure
    """
    horz, horz_atr = readfile.read(str(horz_file))
    vert, vert_atr = readfile.read(str(vert_file))
    
    ref_y = int(horz_atr.get('REF_Y', -1))
    ref_x = int(horz_atr.get('REF_X', -1))
    
    # Auto-compute symmetric color limits if not provided
    if vlim is None:
        max_abs = max(np.nanmax(np.abs(horz)), np.nanmax(np.abs(vert)))
        vlim = (-max_abs, max_abs)
    
    # Handle 3D data (time, lat, lon)
    if horz.ndim == 3:
        n_times = horz.shape[0]
        
        if time_idx is not None:
            # Plot single time step
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            
            im0 = axes[0].imshow(horz[time_idx], cmap=cmap, vmin=vlim[0], vmax=vlim[1])
            axes[0].set_title(f'Horizontal (t={time_idx})')
            plt.colorbar(im0, ax=axes[0], label='m')
            
            im1 = axes[1].imshow(vert[time_idx], cmap=cmap, vmin=vlim[0], vmax=vlim[1])
            axes[1].set_title(f'Vertical (t={time_idx})')
            plt.colorbar(im1, ax=axes[1], label='m')
            
            if ref_y >= 0 and ref_x >= 0:
                for ax in axes:
                    ax.plot(ref_x, ref_y, 'k*', markersize=10)
        else:
            # Plot all time steps
            fig, axes = plt.subplots(2, n_times, figsize=(3 * n_times, 6))
            
            for t in range(n_times):
                axes[0, t].imshow(horz[t], cmap=cmap, vmin=vlim[0], vmax=vlim[1])
                axes[0, t].set_title(f'H t={t}')
                axes[0, t].axis('off')
                
                axes[1, t].imshow(vert[t], cmap=cmap, vmin=vlim[0], vmax=vlim[1])
                axes[1, t].set_title(f'V t={t}')
                axes[1, t].axis('off')
            
            axes[0, 0].set_ylabel('Horizontal')
            axes[1, 0].set_ylabel('Vertical')
    else:
        # 2D data
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        im0 = axes[0].imshow(horz, cmap=cmap, vmin=vlim[0], vmax=vlim[1])
        axes[0].set_title('Horizontal Displacement')
        plt.colorbar(im0, ax=axes[0], label='m')
        
        im1 = axes[1].imshow(vert, cmap=cmap, vmin=vlim[0], vmax=vlim[1])
        axes[1].set_title('Vertical Displacement')
        plt.colorbar(im1, ax=axes[1], label='m')
        
        if ref_y >= 0 and ref_x >= 0:
            for ax in axes:
                ax.plot(ref_x, ref_y, 'k*', markersize=10)
    
    plt.tight_layout()
    plt.show()
    
    return fig

In [ ]:
overlapping_pairs = get_asc_desc_pairs(data_storage)
print(len(overlapping_pairs))

In [ ]:
# TODO: Look into adding processing for the velocity files as well
# TODO: Work on fixing the selection of the ref lat/lon, make sure that the point is within the clipped area. May have something to do with the projection?

# processor = InSARDecomposer(overlapping_pairs, ds_name='timeseries', azimuth=-90)
processor = test_decomposer(overlapping_pairs, ds_name='timeseries', azimuth=-90)
processor.run()

# Access results after processing
print(processor.successful_pairs)
print(processor.failed_pairs)

In [ ]:
# TODO: Ensure that the data is being saved correctly. Some of the plots look like they are the same data. May also be caused by how we are visualizing the data.


test_files = ['/home/clayc/Documents/US_Mex_InSAR/OPERA_data/24725_14877_dhorz.h5', '/home/clayc/Documents/US_Mex_InSAR/OPERA_data/24725_14877_dvert.h5']


# Plot single time step
displacement_fig = plot_displacements(test_files[0], test_files[1], time_idx=4)

In [ ]:
# Plot all time steps (can be large!)
all_disp = plot_displacements(test_files[0], test_files[1])

In [ ]:
# With custom color limits
test_disp_colors = plot_displacements(test_files[0], test_files[1], time_idx=5, vlim=(-0.1, 0.1))